# اجرای ویترین‌یاب در Google Colab\nاین نوت‌بوک Laravel، سرویس FastAPI و SQLite محلی را بدون Docker اجرا می‌کند و در پایان یک لینک HTTPS موقت Cloudflare نمایش می‌دهد.\n\nاگر مخزن GitHub خصوصی است، در Colab از پنل Secrets یک secret با نام `GITHUB_TOKEN` بسازید و دسترسی آن را به این نوت‌بوک بدهید. توکن فقط باید مجوز خواندن همین مخزن را داشته باشد.

In [ ]:
# تنظیم مخزن و کلون امن آن\nREPOSITORY = 'https://github.com/Alirezaab78/clothes_search.git'\nPROJECT_DIR = '/content/clothes_search'\n\nfrom google.colab import userdata\nimport os, subprocess, shutil\n\ntry:\n    token = userdata.get('GITHUB_TOKEN')\nexcept Exception:\n    token = None\n\nif os.path.exists(PROJECT_DIR):\n    shutil.rmtree(PROJECT_DIR)\n\nclone_url = REPOSITORY\nif token:\n    clone_url = REPOSITORY.replace('https://', f'https://x-access-token:{token}@')\n\nsubprocess.run(['git', 'clone', '--depth', '1', clone_url, PROJECT_DIR], check=True)\n# توکن را از remote محلی پاک می‌کنیم تا در فایل config مخزن نماند.\nsubprocess.run(['git', '-C', PROJECT_DIR, 'remote', 'set-url', 'origin', REPOSITORY], check=True)\nprint('✅ Project cloned:', PROJECT_DIR)

In [ ]:
%%bash\nset -euo pipefail\nPROJECT_DIR=/content/clothes_search\nexport DEBIAN_FRONTEND=noninteractive\n\n# PHP، extensionهای Laravel و Composer\napt-get update -qq\napt-get install -y -qq php-cli php-curl php-mbstring php-xml php-zip php-sqlite3 unzip curl git\nif ! command -v composer >/dev/null; then\n  curl -sS https://getcomposer.org/installer -o /tmp/composer-setup.php\n  php /tmp/composer-setup.php --install-dir=/usr/local/bin --filename=composer --quiet\nfi\n\n# محیط پایتون و سرویس هوش مصنوعی\ncd $PROJECT_DIR/ai-service\npython3 -m venv .venv\n. .venv/bin/activate\npython -m pip install --upgrade pip -q\npip install -r requirements.txt -q\nnohup .venv/bin/uvicorn app.main:app --host 127.0.0.1 --port 8001 >/tmp/fashion-ai.log 2>&1 &\n\n# Laravel و SQLite\ncd $PROJECT_DIR/laravel-app\ncomposer install --no-interaction --prefer-dist --optimize-autoloader\ncp -n .env.example .env || true\ntouch database/database.sqlite\nsed -i 's|^DB_CONNECTION=.*|DB_CONNECTION=sqlite|' .env\nsed -i 's|^FASHION_AI_URL=.*|FASHION_AI_URL=http://127.0.0.1:8001|' .env\ngrep -q '^FASHION_AI_URL=' .env || echo 'FASHION_AI_URL=http://127.0.0.1:8001' >> .env\nphp artisan key:generate --force\nphp artisan migrate --force\nphp artisan storage:link || true\nnohup php artisan serve --host=127.0.0.1 --port=8000 >/tmp/laravel.log 2>&1 &\n\n# انتظار برای بالا آمدن هر دو سرویس\nfor i in $(seq 1 90); do curl -fs http://127.0.0.1:8001/health >/dev/null && break; sleep 2; done\nfor i in $(seq 1 30); do curl -fs http://127.0.0.1:8000 >/dev/null && break; sleep 1; done\ncurl -fs http://127.0.0.1:8001/health\necho '\n✅ Laravel and FastAPI are running.'

In [ ]:
%%bash\nset -euo pipefail\n# Cloudflare Quick Tunnel: لینک موقتی HTTPS بدون نیاز به حساب Cloudflare\ncurl -L --retry 3 -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\nchmod +x /usr/local/bin/cloudflared\nnohup cloudflared tunnel --url http://127.0.0.1:8000 >/tmp/cloudflared.log 2>&1 &\n\nfor i in $(seq 1 30); do\n  URL=$(grep -oE 'https://[-a-z0-9]+\.trycloudflare\.com' /tmp/cloudflared.log | head -n 1 || true)\n  [ -n "$URL" ] && break\n  sleep 2\ndone\nif [ -z "${URL:-}" ]; then\n  echo 'Tunnel URL was not created. See the log below:'\n  cat /tmp/cloudflared.log\n  exit 1\nfi\necho '🎉 Public URL (temporary):'\necho "$URL"

## نکات\n- لینک `trycloudflare.com` موقتی است و تا زمان فعال بودن Runtime کولب معتبر می‌ماند.\n- پس از ری‌استارت Runtime، سلول‌های کلون، راه‌اندازی و تونل را دوباره اجرا کنید.\n- دانلود نخست مدل OpenCLIP ممکن است زمان‌بر باشد.\n- برای مخزن عمومی، نیازی به `GITHUB_TOKEN` نیست.